In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1: ENVIRONMENT SETUP                                       ║
# ║  Run this FIRST. Press Shift+Enter. Wait for the ✅ message.     ║
# ║  If you see a red error, just run it again — Colab sometimes     ║
# ║  needs a second attempt on fresh installs.                       ║
# ╚══════════════════════════════════════════════════════════════════╝

!pip install yfinance openpyxl plotly scipy --quiet

# --- Standard libraries (already in Colab, just importing) ---
import pandas as pd           # Think of this as Excel tables in Python
import numpy as np            # Math library
import warnings
warnings.filterwarnings('ignore')

# --- Data pulling ---
import yfinance as yf          # Pulls real financials from Yahoo Finance (FREE)

# --- Charting ---
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# --- Excel output ---
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# --- Math solver (for the debt capacity optimizer) ---
from scipy.optimize import brentq

# --- Colab file download utility ---
from google.colab import files

print("✅ All libraries loaded. You're ready to build.")
print("   Python version ready. Colab environment confirmed.")

✅ All libraries loaded. You're ready to build.
   Python version ready. Colab environment confirmed.


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2: COMPANY DEFINITIONS & ASSUMPTIONS                       ║
# ║                                                                  ║
# ║  We define BOTH companies here — Lockheed Martin (LMT) and      ║
# ║  Palantir (PLTR) — along with their sector-specific LBO         ║
# ║  assumptions, projection inputs, and analyst commentary.         ║
# ║                                                                  ║
# ║  IMPORTANT ANALYST NOTE (read before your GitHub README):       ║
# ║  LMT  → Realistic LBO candidate. Stable US gov revenue,         ║
# ║          FCF of $6.9B (FY2025), existing leverage ~2x.          ║
# ║          Source: LMT FY2025 Annual Report (Jan 2026)             ║
# ║  PLTR → NOT a realistic LBO target. EV/EBITDA ~137x means       ║
# ║          you cannot buy it at a price that makes debt            ║
# ║          service economically viable. Included as a              ║
# ║          credit quality benchmark to demonstrate the             ║
# ║          key LevFin insight: high FCF margin ≠ LBO feasible.    ║
# ║          Source: PLTR FY2025 earnings (stockanalysis.com)        ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── CURRENT MARKET RATES (justify these if asked in interview) ─────
SOFR_RATE    = 0.043   # US SOFR ~4.3% as of mid-2026 (declining from 2024 peak)
TLB_SPREAD   = 0.0350  # 350bps — typical for B2/B rated LBO (2025 market)
                        # Source: Baker McKenzie Lev Finance Report 2025
HY_RATE      = 0.0950  # 9.50% fixed — AFME HY Report Q2 2025
SSN_RATE     = 0.0875  # 8.75% — Senior Secured Notes (tighter than HY unsecured)
MEZZ_RATE    = 0.1300  # 13.00% — Standard mezzanine pricing

# ── COMPANY METADATA ───────────────────────────────────────────────
COMPANIES = {
    'LMT': {
        'name':   'Lockheed Martin Corp.',
        'sector': 'Aerospace & Defense',
        'ticker': 'LMT',

        # ── VERIFIED FY2025 FINANCIALS ($M) ─────────────────────
        # Source: LMT Q4 2025 Earnings Release, Jan 28 2026
        # Revenue $75.0B, EBITDA $9.4B, FCF $6.9B
        'override_financials': {
            'base_revenue':      75_000,   # $75.0B in $M
            'base_ebitda':        9_400,   # $9.4B
            'base_da':            1_200,   # D&A ~$1.2B
            'base_capex':         1_800,   # CapEx ~$1.8B (defense = asset heavy)
            'base_fcf':           6_900,   # FCF $6.9B (cash ops $8.6B - capex)
            'existing_debt':     21_500,   # ~$21.5B existing debt
        },

        # ── 5-YEAR PROJECTION ASSUMPTIONS ───────────────────────
        # LMT: Revenue grows with US defense budget (~4-6% CAGR)
        # Margins stable — cost-plus contracts limit upside/downside
        'projections': {
            'revenue_growth':    [0.05, 0.05, 0.04, 0.04, 0.03],
            'ebitda_margin':     [0.125, 0.128, 0.130, 0.132, 0.133],
            'capex_pct_rev':     [0.024, 0.024, 0.023, 0.023, 0.022],
            'tax_rate':          0.16,   # LMT effective rate (R&D credits reduce it)
            'nwc_pct_rev':       0.04,   # Defense: low NWC intensity (advance payments)
        },

        # ── LENDER THRESHOLDS (defense/industrial sector norms) ─
        'covenants': {
            'min_dscr':   1.20,   # 1.20x minimum DSCR
            'min_icr':    2.50,   # 2.50x interest coverage
            'max_lev':    5.50,   # 5.5x max entry leverage (defense = predictable)
        },

        # ── CAPITAL STRUCTURES TO TEST ──────────────────────────
        # Each option is (tranche: leverage multiple vs entry EBITDA)
        'structures': {
            'Option A — TLB Heavy':    {'TLB': 3.5, 'HY':  1.0},
            'Option B — Balanced':     {'TLB': 2.5, 'SSN': 1.5, 'Mezz': 0.5},
            'Option C — Bond Heavy':   {'TLB': 2.0, 'HY':  2.5},
        },

        'lbo_viable': True,
        'lbo_note': (
            "LBO FEASIBLE. Government cost-plus contracts provide highly "
            "predictable revenue (F-35 program alone = ~$10B+ annual revenue). "
            "FCF of $6.9B (FY2025) comfortably covers debt service across structures. "
            "Primary risk: pension obligations (~$15B unfunded) and contract renewals."
        ),
    },

    'PLTR': {
        'name':   'Palantir Technologies Inc.',
        'sector': 'Enterprise Software / AI',
        'ticker': 'PLTR',

        # ── VERIFIED FY2025 FINANCIALS ($M) ─────────────────────
        # Source: PLTR FY2025 (stockanalysis.com, monexa.ai)
        # Revenue ~$3.9B, EBITDA ~$2.0B, FCF ~$1.8B, ZERO debt
        'override_financials': {
            'base_revenue':       3_900,
            'base_ebitda':        2_018,   # Per stockanalysis.com FY2025
            'base_da':              120,   # D&A minimal (software = few physical assets)
            'base_capex':            40,   # CapEx ~$40M — capital-light
            'base_fcf':           1_800,   # FCF ~$1.8B (51% FCF margin)
            'existing_debt':          0,   # DEBT FREE
        },

        # ── 5-YEAR PROJECTION ASSUMPTIONS ───────────────────────
        # PLTR: US commercial revenue +93% YoY in Q2 2025
        # Growth decelerates but stays high (software scaling)
        'projections': {
            'revenue_growth':    [0.40, 0.32, 0.25, 0.20, 0.18],
            'ebitda_margin':     [0.40, 0.42, 0.44, 0.45, 0.46],
            'capex_pct_rev':     [0.010, 0.010, 0.010, 0.009, 0.009],
            'tax_rate':          0.15,   # Low effective rate (stock comp deductions)
            'nwc_pct_rev':       0.05,
        },

        # ── LENDER THRESHOLDS (software sector — tighter on leverage) ─
        'covenants': {
            'min_dscr':   1.20,
            'min_icr':    3.00,   # Software lenders demand higher ICR (growth risk)
            'max_lev':    3.00,   # 3x max — software cash flows less "certain"
        },

        # ── CONSERVATIVE HYPOTHETICAL STRUCTURES ────────────────
        # Constrained by realistic lender appetite for software LBOs
        'structures': {
            'Option A — Light Leverage': {'TLB': 1.5, 'SSN': 0.5},
            'Option B — Moderate':       {'TLB': 2.0, 'HY':  0.75},
        },

        'lbo_viable': False,
        'lbo_note': (
            "⚠️  LBO NOT FEASIBLE AT CURRENT VALUATION. "
            "EV/EBITDA ~137x implies a purchase price of ~$276B for $2B of EBITDA. "
            "At 5x leverage that's only $10B of debt on a $276B deal — "
            "the equity check is ~$266B, making sponsor returns impossible. "
            "Analysis below is HYPOTHETICAL ONLY to demonstrate credit quality. "
            "A LevFin analyst would decline to underwrite this at current pricing."
        ),
    },
}

print("✅ Company definitions loaded.")
for ticker, co in COMPANIES.items():
    fin = co['override_financials']
    print(f"\n  {co['name']} ({ticker})")
    print(f"    Revenue:  ${fin['base_revenue']:>8,.0f}M")
    print(f"    EBITDA:   ${fin['base_ebitda']:>8,.0f}M  "
          f"(margin: {fin['base_ebitda']/fin['base_revenue']:.1%})")
    print(f"    FCF:      ${fin['base_fcf']:>8,.0f}M  "
          f"(margin: {fin['base_fcf']/fin['base_revenue']:.1%})")
    print(f"    Debt:     ${fin['existing_debt']:>8,.0f}M")
    print(f"    LBO:      {'✅ Feasible' if co['lbo_viable'] else '❌ NOT feasible'}")

✅ Company definitions loaded.

  Lockheed Martin Corp. (LMT)
    Revenue:  $  75,000M
    EBITDA:   $   9,400M  (margin: 12.5%)
    FCF:      $   6,900M  (margin: 9.2%)
    Debt:     $  21,500M
    LBO:      ✅ Feasible

  Palantir Technologies Inc. (PLTR)
    Revenue:  $   3,900M
    EBITDA:   $   2,018M  (margin: 51.7%)
    FCF:      $   1,800M  (margin: 46.2%)
    Debt:     $       0M
    LBO:      ❌ NOT feasible


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3: LIVE DATA PULL + VALIDATION                             ║
# ║                                                                  ║
# ║  We pull live data from yfinance to validate our hard-coded     ║
# ║  figures. If yfinance data matches our overrides within ±15%,   ║
# ║  we use our overrides (more accurate, from primary sources).    ║
# ║  If yfinance shows a big discrepancy, we flag it.               ║
# ║                                                                  ║
# ║  WHY hard-code AND pull live data?                               ║
# ║  Because yfinance sometimes has stale or misclassified data.    ║
# ║  Professionals always cross-reference. So do we.                ║
# ╚══════════════════════════════════════════════════════════════════╝

def safe_get(df, possible_names, default=0):
    """
    Safely retrieves a column from a DataFrame by trying multiple
    possible column names. yfinance is inconsistent with naming.
    Returns a Series of the first match, or a zero Series if none found.
    """
    for name in possible_names:
        if name in df.columns:
            val = df[name].fillna(0)
            return val
    return pd.Series([default] * len(df), index=df.index)

def pull_and_validate(ticker_sym, company_def):
    """
    Pulls historical financials from yfinance, computes key metrics,
    and validates against our hard-coded override figures.

    Returns a DataFrame of historical data in $M.
    """
    print(f"\n{'─'*60}")
    print(f"  Pulling: {company_def['name']} ({ticker_sym})")
    print(f"{'─'*60}")

    try:
        obj = yf.Ticker(ticker_sym)
        inc = obj.financials.T.sort_index()
        bs  = obj.balance_sheet.T.sort_index()
        cf  = obj.cashflow.T.sort_index()

        # Extract line items
        rev     = safe_get(inc, ['Total Revenue', 'Revenue'])
        ebit    = safe_get(inc, ['EBIT', 'Operating Income'])
        int_exp = safe_get(inc, ['Interest Expense',
                                  'Interest Expense Non Operating']).abs()
        da      = safe_get(cf,  ['Depreciation And Amortization',
                                  'Reconciled Depreciation']).abs()
        capex   = safe_get(cf,  ['Capital Expenditure',
                                  'Purchase Of PPE']).abs()
        delta_wc = safe_get(cf, ['Change In Working Capital']).fillna(0)
        tax     = safe_get(inc, ['Tax Provision']).abs()
        t_debt  = safe_get(bs,  ['Total Debt',
                                  'Long Term Debt And Capital Lease Obligation'])
        cash    = safe_get(bs,  ['Cash And Cash Equivalents',
                                  'Cash Cash Equivalents And Short Term Investments'])

        ebitda  = ebit + da
        fcf     = ebitda - capex - tax - delta_wc.abs()
        net_dbt = t_debt - cash

        hist = pd.DataFrame({
            'Revenue':         rev,
            'EBITDA':          ebitda,
            'EBIT':            ebit,
            'DA':              da,
            'CapEx':           capex,
            'FCF':             fcf,
            'Interest_Exp':    int_exp,
            'Total_Debt':      t_debt,
            'Cash':            cash,
            'Net_Debt':        net_dbt,
            'Tax':             tax,
        }).dropna(how='all') / 1e6   # Convert to $M

        hist['EBITDA_Margin'] = hist['EBITDA'] / hist['Revenue']
        hist['FCF_Margin']    = hist['FCF']    / hist['Revenue']
        hist['Leverage']      = hist['Total_Debt'] / hist['EBITDA']

        # ── VALIDATION vs. hard-coded overrides ─────────────────
        ov = company_def['override_financials']
        if len(hist) > 0:
            yf_rev   = hist['Revenue'].iloc[-1]
            yf_ebitda = hist['EBITDA'].iloc[-1]
            ov_rev   = ov['base_revenue']
            ov_ebitda = ov['base_ebitda']

            rev_diff   = abs(yf_rev - ov_rev) / ov_rev
            ebitda_diff = abs(yf_ebitda - ov_ebitda) / ov_ebitda

            print(f"  yfinance Revenue: ${yf_rev:>8,.0f}M  "
                  f"| Override: ${ov_rev:>8,.0f}M  "
                  f"| Diff: {rev_diff:.1%} "
                  f"{'✅' if rev_diff < 0.15 else '⚠️ CHECK SOURCE'}")
            print(f"  yfinance EBITDA:  ${yf_ebitda:>8,.0f}M  "
                  f"| Override: ${ov_ebitda:>8,.0f}M  "
                  f"| Diff: {ebitda_diff:.1%} "
                  f"{'✅' if ebitda_diff < 0.20 else '⚠️ CHECK SOURCE'}")
        else:
            print("  ⚠️  No yfinance data returned — using override figures only.")

        print(f"  Historical years available: {[y.year for y in hist.index]}")
        return hist

    except Exception as e:
        print(f"  ❌ yfinance pull failed: {e}")
        print("  → Falling back to override figures entirely. This is fine.")
        return None

# ── RUN FOR BOTH COMPANIES ──────────────────────────────────────────
live_data = {}
for ticker, co_def in COMPANIES.items():
    live_data[ticker] = pull_and_validate(ticker, co_def)

print("\n\n✅ Data pull complete. Proceeding with override figures (primary sources).")


────────────────────────────────────────────────────────────
  Pulling: Lockheed Martin Corp. (LMT)
────────────────────────────────────────────────────────────
  yfinance Revenue: $  75,048M  | Override: $  75,000M  | Diff: 0.1% ✅
  yfinance EBITDA:  $   8,727M  | Override: $   9,400M  | Diff: 7.2% ✅
  Historical years available: [2021, 2022, 2023, 2024, 2025]

────────────────────────────────────────────────────────────
  Pulling: Palantir Technologies Inc. (PLTR)
────────────────────────────────────────────────────────────
  yfinance Revenue: $   4,475M  | Override: $   3,900M  | Diff: 14.8% ✅
  yfinance EBITDA:  $   1,440M  | Override: $   2,018M  | Diff: 28.6% ⚠️ CHECK SOURCE
  Historical years available: [2021, 2022, 2023, 2024, 2025]


✅ Data pull complete. Proceeding with override figures (primary sources).


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4: 5-YEAR LBO PROJECTION ENGINE                            ║
# ║                                                                  ║
# ║  Builds a forward P&L and cash flow model for any company       ║
# ║  given its base year financials and growth assumptions.          ║
# ║                                                                  ║
# ║  What this cell produces:                                        ║
# ║  A table like this for each company, each year:                  ║
# ║    Year | Revenue | EBITDA | CapEx | Tax | UFCF                  ║
# ║  UFCF = Unlevered Free Cash Flow = cash BEFORE debt payments.   ║
# ║  This is what the company uses to pay its lenders.               ║
# ╚══════════════════════════════════════════════════════════════════╝

N_YEARS = 5   # 5-year hold period (standard LBO assumption)

def build_lbo_projection(company_def, scenario_overrides=None):
    """
    Constructs a 5-year income statement + cash flow projection
    for a given company under given assumptions.

    Args:
        company_def:       Company dict from COMPANIES
        scenario_overrides: Optional dict to override growth/margin assumptions
                            (used in scenario analysis). None = base case.

    Returns:
        DataFrame indexed Y1–Y5 with P&L and FCF line items.
    """
    fin  = company_def['override_financials']
    proj = company_def['projections'].copy()

    # Apply scenario overrides if provided (used in stress testing)
    if scenario_overrides:
        for key, val in scenario_overrides.items():
            proj[key] = val

    base_rev  = fin['base_revenue']
    base_da   = fin['base_da']

    rows = {}
    prev_nwc = base_rev * proj['nwc_pct_rev']

    for i in range(N_YEARS):
        yr_label = f'Y{i+1}'

        # Revenue: grow from prior year
        if i == 0:
            rev = base_rev * (1 + proj['revenue_growth'][i])
        else:
            rev = rows[f'Y{i}']['Revenue'] * (1 + proj['revenue_growth'][i])

        # EBITDA: apply margin to revenue
        ebitda   = rev * proj['ebitda_margin'][i]

        # D&A: scale proportionally with revenue
        da_est   = base_da * (rev / base_rev)

        # EBIT: EBITDA minus D&A
        ebit     = ebitda - da_est

        # CapEx: expressed as % of revenue
        capex    = rev * proj['capex_pct_rev'][i]

        # Working capital: change in NWC (cash outflow when growing)
        curr_nwc = rev * proj['nwc_pct_rev']
        delta_nwc = curr_nwc - prev_nwc   # Positive = cash absorbed
        prev_nwc = curr_nwc

        # Tax: applied on EBIT (pre-interest, unlevered basis)
        tax = max(ebit * proj['tax_rate'], 0)

        # UFCF = cash available BEFORE any debt payments
        # This is what gets split between lenders and equity
        ufcf = ebitda - capex - tax - delta_nwc

        rows[yr_label] = {
            'Year':          yr_label,
            'Revenue':       round(rev, 1),
            'EBITDA':        round(ebitda, 1),
            'EBITDA_Margin': ebitda / rev,
            'DA':            round(da_est, 1),
            'EBIT':          round(ebit, 1),
            'CapEx':         round(capex, 1),
            'Delta_NWC':     round(delta_nwc, 1),
            'Tax':           round(tax, 1),
            'UFCF':          round(ufcf, 1),
        }

    df = pd.DataFrame(rows).T.set_index('Year')
    return df.apply(pd.to_numeric, errors='ignore')


# ── BUILD AND DISPLAY BASE CASE PROJECTIONS FOR BOTH COMPANIES ─────
projections = {}
for ticker, co_def in COMPANIES.items():
    proj_df = build_lbo_projection(co_def)
    projections[ticker] = proj_df

    print(f"\n{'═'*65}")
    print(f"  {co_def['name']} ({ticker}) — 5-Year Base Case Projection ($M)")
    print(f"{'═'*65}")
    display(proj_df[['Revenue','EBITDA','EBITDA_Margin',
                      'CapEx','UFCF']].style.format({
        'Revenue':       '${:>8,.0f}M',
        'EBITDA':        '${:>8,.0f}M',
        'EBITDA_Margin': '{:.1%}',
        'CapEx':         '${:>8,.0f}M',
        'UFCF':          '${:>8,.0f}M',
    }))

print("\n✅ Projections built. UFCF = cash available to service debt.")


═════════════════════════════════════════════════════════════════
  Lockheed Martin Corp. (LMT) — 5-Year Base Case Projection ($M)
═════════════════════════════════════════════════════════════════


,Revenue,EBITDA,EBITDA_Margin,CapEx,UFCF
Year,,,,,
Y1,"$ 78,750M","$ 9,844M",12.5%,"$ 1,890M","$ 6,430M"
Y2,"$ 82,688M","$ 10,584M",12.8%,"$ 1,984M","$ 6,960M"
Y3,"$ 85,995M","$ 11,179M",13.0%,"$ 1,978M","$ 7,501M"
Y4,"$ 89,435M","$ 11,805M",13.2%,"$ 2,057M","$ 7,951M"
Y5,"$ 92,118M","$ 12,252M",13.3%,"$ 2,027M","$ 8,393M"



═════════════════════════════════════════════════════════════════
  Palantir Technologies Inc. (PLTR) — 5-Year Base Case Projection ($M)
═════════════════════════════════════════════════════════════════


,Revenue,EBITDA,EBITDA_Margin,CapEx,UFCF
Year,,,,,
Y1,"$ 5,460M","$ 2,184M",40.0%,$ 55M,"$ 1,749M"
Y2,"$ 7,207M","$ 3,027M",42.0%,$ 72M,"$ 2,447M"
Y3,"$ 9,009M","$ 3,964M",44.0%,$ 90M,"$ 3,231M"
Y4,"$ 10,811M","$ 4,865M",45.0%,$ 97M,"$ 3,998M"
Y5,"$ 12,757M","$ 5,868M",46.0%,$ 115M,"$ 4,835M"



✅ Projections built. UFCF = cash available to service debt.


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5: DEBT TRANCHE & CAPITAL STRUCTURE MODULE                 ║
# ║                                                                  ║
# ║  Defines each piece of debt in the capital structure.            ║
# ║  Think of this like a list of loans, each with its own:         ║
# ║    - Size (how much you borrow)                                  ║
# ║    - Interest rate (how much you pay per year)                   ║
# ║    - Amortization (how much principal you repay per year)        ║
# ║    - Maturity (when the rest is due in a lump sum)               ║
# ║    - Seniority (who gets paid first if things go wrong)          ║
# ║                                                                  ║
# ║  MARKET PRICING JUSTIFICATION:                                   ║
# ║  TLB:  SOFR (4.3%) + 350bps = 7.8% all-in (2025 market)        ║
# ║        Source: Baker McKenzie LevFin Annual 2025, S&P Global     ║
# ║  SSN:  8.75% fixed — senior secured, 1st lien bonds             ║
# ║  HY:   9.50% fixed — AFME European HY Report Q2 2025            ║
# ║  Mezz: 13.00% — standard subordinated mezzanine                 ║
# ╚══════════════════════════════════════════════════════════════════╝

def create_tranche(name, principal_m, rate, amort_pct,
                   tenor_yrs, rank, is_pik=False, is_floating=False,
                   base_rate=SOFR_RATE, spread=0.0):
    """
    Creates a single debt tranche dictionary.

    Args:
        name:         Human-readable tranche name
        principal_m:  Drawn amount in $M
        rate:         All-in rate (fixed) OR spread (if floating)
        amort_pct:    Annual mandatory amortization as % of original principal
        tenor_yrs:    Years until bullet maturity
        rank:         Seniority rank (1 = most senior, highest priority in default)
        is_pik:       PIK = Pay-In-Kind. Interest adds to principal, no cash today.
        is_floating:  True = SOFR-based floating rate
        base_rate:    Base rate if floating (SOFR)
        spread:       Credit spread over base rate (e.g., 0.035 = 350bps)
    """
    if is_floating:
        all_in = base_rate + spread
    else:
        all_in = rate

    annual_cash_int = 0 if is_pik else principal_m * all_in
    annual_amort_m  = principal_m * amort_pct

    return {
        'Name':              name,
        'Principal':         principal_m,
        'All_In_Rate':       all_in,
        'Is_Floating':       is_floating,
        'Is_PIK':            is_pik,
        'Amort_Pct':         amort_pct,
        'Annual_Amort':      annual_amort_m,
        'Annual_Cash_Int':   annual_cash_int,
        'Tenor':             tenor_yrs,
        'Rank':              rank,
    }


def assemble_capital_structure(entry_ebitda, structure_config):
    """
    Builds a list of tranche objects from a structure config dict.

    Args:
        entry_ebitda:     Year 1 EBITDA ($M) — used to size tranches
        structure_config: Dict mapping tranche type to leverage multiple
                          e.g., {'TLB': 3.5, 'HY': 1.0}

    Returns:
        List of tranche dicts, sorted by rank (senior first)
    """
    tranches = []

    if 'TLB' in structure_config:
        size = entry_ebitda * structure_config['TLB']
        tranches.append(create_tranche(
            name         = f'Term Loan B  ({structure_config["TLB"]:.1f}x)',
            principal_m  = size,
            rate         = 0,                 # Floating — rate set below
            amort_pct    = 0.01,              # 1% per annum (LBO market standard)
            tenor_yrs    = 7,
            rank         = 1,
            is_floating  = True,
            base_rate    = SOFR_RATE,
            spread       = TLB_SPREAD,        # SOFR + 350bps
        ))

    if 'SSN' in structure_config:
        size = entry_ebitda * structure_config['SSN']
        tranches.append(create_tranche(
            name         = f'Senior Secured Notes  ({structure_config["SSN"]:.1f}x)',
            principal_m  = size,
            rate         = SSN_RATE,          # 8.75% fixed
            amort_pct    = 0.0,               # Bullet — no annual amort
            tenor_yrs    = 8,
            rank         = 2,
        ))

    if 'HY' in structure_config:
        size = entry_ebitda * structure_config['HY']
        tranches.append(create_tranche(
            name         = f'High Yield Notes  ({structure_config["HY"]:.1f}x)',
            principal_m  = size,
            rate         = HY_RATE,           # 9.50% fixed
            amort_pct    = 0.0,               # Bullet
            tenor_yrs    = 8,
            rank         = 3,
        ))

    if 'Mezz' in structure_config:
        size = entry_ebitda * structure_config['Mezz']
        tranches.append(create_tranche(
            name         = f'Mezzanine  ({structure_config["Mezz"]:.1f}x)',
            principal_m  = size,
            rate         = MEZZ_RATE,         # 13.00%
            amort_pct    = 0.0,
            tenor_yrs    = 9,
            rank         = 4,
            is_pik       = False,             # Set True to model PIK toggle
        ))

    return sorted(tranches, key=lambda x: x['Rank'])


# ── BUILD ALL STRUCTURES FOR BOTH COMPANIES ─────────────────────────
all_structures = {}

for ticker, co_def in COMPANIES.items():
    entry_ebitda = projections[ticker].iloc[0]['EBITDA']
    all_structures[ticker] = {}

    print(f"\n{'═'*65}")
    print(f"  {co_def['name']} ({ticker})")
    print(f"  Entry EBITDA: ${entry_ebitda:,.0f}M")
    print(f"{'═'*65}")

    for struct_name, config in co_def['structures'].items():
        tranches = assemble_capital_structure(entry_ebitda, config)
        all_structures[ticker][struct_name] = tranches

        total_d = sum(t['Principal'] for t in tranches)
        total_i = sum(t['Annual_Cash_Int'] for t in tranches)
        blended = total_i / total_d if total_d > 0 else 0

        print(f"\n  {struct_name}")
        print(f"    {'Tranche':<35} {'Size ($M)':>10} {'Rate':>8} {'Amort':>8} {'Tenor':>7}")
        print(f"    {'─'*68}")
        for t in tranches:
            print(f"    {t['Name']:<35} "
                  f"${t['Principal']:>8,.0f}M  "
                  f"{t['All_In_Rate']:>7.2%}  "
                  f"{t['Amort_Pct']:>7.0%}  "
                  f"{t['Tenor']:>5}yr")
        print(f"    {'─'*68}")
        print(f"    {'TOTAL':<35} ${total_d:>8,.0f}M  "
              f"{'Blended:':>8} {blended:>6.2%}")
        print(f"    Leverage: {total_d/entry_ebitda:.1f}x EBITDA")

print("\n✅ Capital structures assembled.")


═════════════════════════════════════════════════════════════════
  Lockheed Martin Corp. (LMT)
  Entry EBITDA: $9,844M
═════════════════════════════════════════════════════════════════

  Option A — TLB Heavy
    Tranche                              Size ($M)     Rate    Amort   Tenor
    ────────────────────────────────────────────────────────────────────
    Term Loan B  (3.5x)                 $  34,453M    7.80%       1%      7yr
    High Yield Notes  (1.0x)            $   9,844M    9.50%       0%      8yr
    ────────────────────────────────────────────────────────────────────
    TOTAL                               $  44,297M  Blended:  8.18%
    Leverage: 4.5x EBITDA

  Option B — Balanced
    Tranche                              Size ($M)     Rate    Amort   Tenor
    ────────────────────────────────────────────────────────────────────
    Term Loan B  (2.5x)                 $  24,610M    7.80%       1%      7yr
    Senior Secured Notes  (1.5x)        $  14,766M    8.75%      

In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6: CREDIT STATISTICS ENGINE + DEBT CAPACITY SOLVER        ║
# ║                                                                  ║
# ║  This is the CORE of the entire project.                         ║
# ║                                                                  ║
# ║  Part A: compute_credit_stats()                                  ║
# ║    → Given a projection and a set of debt tranches,              ║
# ║      calculates DSCR, ICR, FCCR, and leverage year by year.     ║
# ║                                                                  ║
# ║  Part B: solve_max_debt()                                        ║
# ║    → Uses scipy's brentq root-finder to answer:                  ║
# ║      "What is the absolute maximum I can borrow on a single     ║
# ║       TLB tranche such that DSCR never falls below 1.20x?"      ║
# ║    → This is literally how LevFin analysts size debt.            ║
# ╚══════════════════════════════════════════════════════════════════╝

def compute_credit_stats(projection_df, tranches):
    """
    Calculates annual credit metrics for one projection + capital structure.

    The debt balance amortizes each year. Bullet tranches repay in full
    at their tenor year. We track outstanding balances dynamically.

    Args:
        projection_df: Output of build_lbo_projection()
        tranches:      List of tranche dicts from assemble_capital_structure()

    Returns:
        DataFrame of annual DSCR, ICR, FCCR, leverage (indexed Y1–Y5)
    """
    stats_rows = []

    # Track each tranche's outstanding balance (reduces with amortization)
    balances = {t['Name']: t['Principal'] for t in tranches}

    for i, (yr, row) in enumerate(projection_df.iterrows()):
        year_num = i + 1

        ebitda = row['EBITDA']
        ebit   = row['EBIT']
        ufcf   = row['UFCF']
        capex  = row['CapEx']

        # ── ANNUAL DEBT SERVICE CALCULATION ─────────────────────
        cash_interest = 0
        mandatory_amort = 0
        pik_accrual = 0
        total_debt = sum(balances.values())

        for t in tranches:
            bal = balances[t['Name']]
            if bal <= 0:
                continue

            # Interest calculation
            yr_interest = bal * t['All_In_Rate']

            if t['Is_PIK']:
                pik_accrual += yr_interest     # Accrues to principal, no cash
            else:
                cash_interest += yr_interest   # Cash outflow

            # Mandatory amortization (min of scheduled vs. remaining balance)
            if year_num == t['Tenor']:
                # Bullet year: repay full outstanding balance
                mandatory_amort += bal
            else:
                sched_amort = t['Annual_Amort']
                mandatory_amort += min(sched_amort, bal)

        # Total cash debt service = interest paid + principal repaid
        total_debt_service = cash_interest + mandatory_amort

        # ── CREDIT RATIOS ────────────────────────────────────────

        # DSCR: UFCF ÷ Total Debt Service
        # Minimum acceptable: 1.20x (standard LBO covenant)
        dscr = ufcf / total_debt_service if total_debt_service > 0 else np.inf

        # ICR: EBIT ÷ Cash Interest
        # Measures whether operating profit covers interest alone
        icr  = ebit / cash_interest if cash_interest > 0 else np.inf

        # FCCR: (EBITDA - CapEx) ÷ (Cash Interest + Mandatory Amort)
        # More conservative than DSCR — captures capex burden too
        fccr = (ebitda - capex) / total_debt_service if total_debt_service > 0 else np.inf

        # Leverage: Total Debt ÷ EBITDA
        leverage = total_debt / ebitda if ebitda > 0 else np.inf

        # Senior leverage (Rank 1 tranches only)
        senior_debt = sum(b for t, b in zip(tranches, balances.values())
                          if t['Rank'] == 1)
        senior_lev  = senior_debt / ebitda if ebitda > 0 else np.inf

        stats_rows.append({
            'Year':           yr,
            'EBITDA':         ebitda,
            'UFCF':           ufcf,
            'Cash_Interest':  cash_interest,
            'Amortization':   mandatory_amort,
            'Debt_Service':   total_debt_service,
            'PIK_Accrual':    pik_accrual,
            'Total_Debt':     total_debt,
            'DSCR':           round(dscr, 3),
            'ICR':            round(icr, 3),
            'FCCR':           round(fccr, 3),
            'Total_Leverage': round(leverage, 3),
            'Senior_Leverage':round(senior_lev, 3),
        })

        # ── UPDATE BALANCES FOR NEXT YEAR ────────────────────────
        for t in tranches:
            bal = balances[t['Name']]
            if bal <= 0:
                balances[t['Name']] = 0
                continue
            if year_num == t['Tenor']:
                balances[t['Name']] = 0       # Bullet repaid
            else:
                amort = min(t['Annual_Amort'], bal)
                pik   = bal * t['All_In_Rate'] if t['Is_PIK'] else 0
                balances[t['Name']] = max(0, bal - amort + pik)

    return pd.DataFrame(stats_rows).set_index('Year')


def solve_max_tlb_capacity(projection_df, min_dscr=1.20, min_icr=2.50):
    """
    Binary-search solver for maximum TLB principal that satisfies
    DSCR and ICR covenants across all 5 projection years.

    Uses scipy's brentq algorithm (efficient bracket root-finding).
    This is mathematically equivalent to how LevFin analysts
    back-solve for debt capacity in Excel using Goal Seek.

    Args:
        projection_df: 5-year projection DataFrame
        min_dscr:      Minimum DSCR threshold (e.g., 1.20)
        min_icr:       Minimum ICR threshold (e.g., 2.50)

    Returns:
        Maximum TLB principal in $M
    """
    def constraint(principal):
        """
        Returns positive value if covenants are SATISFIED,
        negative if BREACHED. Zero = exactly at threshold.
        brentq finds the zero of this function.
        """
        t = create_tranche(
            name        = 'TLB_Test',
            principal_m = principal,
            rate        = 0,
            amort_pct   = 0.01,
            tenor_yrs   = 7,
            rank        = 1,
            is_floating = True,
            base_rate   = SOFR_RATE,
            spread      = TLB_SPREAD,
        )
        stats = compute_credit_stats(projection_df, [t])

        # Find worst-case ratio across all years
        min_dscr_achieved = stats['DSCR'].min()
        min_icr_achieved  = stats['ICR'].min()

        # How far are we from the thresholds? (positive = OK, negative = breach)
        dscr_slack = min_dscr_achieved - min_dscr
        icr_slack  = min_icr_achieved  - min_icr

        return min(dscr_slack, icr_slack)  # Binding constraint

    # Define search range: $1M to 20x max EBITDA
    max_search = projection_df['EBITDA'].max() * 20

    try:
        if constraint(1.0) < 0:
            return 0.0   # Can't support even $1M of debt

        if constraint(max_search) > 0:
            return max_search  # Unconstrained within search range

        # brentq finds the exact zero of constraint() in [1, max_search]
        max_principal = brentq(constraint, 1.0, max_search, xtol=1.0)
        return round(max_principal, 0)

    except Exception as e:
        print(f"    Solver error: {e}")
        return 0.0


# ── RUN CREDIT STATS FOR ALL STRUCTURES ──────────────────────────────
all_credit_stats = {}

for ticker, co_def in COMPANIES.items():
    all_credit_stats[ticker] = {}
    cov = co_def['covenants']

    print(f"\n{'═'*70}")
    print(f"  CREDIT STATISTICS — {co_def['name']} ({ticker})")
    print(f"  Covenants: DSCR ≥ {cov['min_dscr']}x | ICR ≥ {cov['min_icr']}x | "
          f"Max Leverage {cov['max_lev']}x")
    print(f"{'═'*70}")

    for struct_name, tranches in all_structures[ticker].items():
        stats = compute_credit_stats(projections[ticker], tranches)
        all_credit_stats[ticker][struct_name] = stats

        print(f"\n  {struct_name}")

        def flag(val, threshold, higher_is_better=True):
            if higher_is_better:
                return '🔴' if val < threshold else '🟢'
            else:
                return '🔴' if val > threshold else '🟢'

        display(stats[['EBITDA','Total_Debt','DSCR','ICR',
                        'FCCR','Total_Leverage','Senior_Leverage']].style
            .format({
                'EBITDA':          '${:>,.0f}M',
                'Total_Debt':      '${:>,.0f}M',
                'DSCR':            '{:.2f}x',
                'ICR':             '{:.2f}x',
                'FCCR':            '{:.2f}x',
                'Total_Leverage':  '{:.2f}x',
                'Senior_Leverage': '{:.2f}x',
            })
            .applymap(
                lambda v: 'background-color:#ffcccc; color:black'
                if isinstance(v, (int,float)) and v < cov['min_dscr'] else '',
                subset=['DSCR']
            )
            .applymap(
                lambda v: 'background-color:#ffcccc; color:black'
                if isinstance(v, (int,float)) and v < cov['min_icr'] else '',
                subset=['ICR']
            )
        )

    # ── DEBT CAPACITY SOLVER ──────────────────────────────────────
    print(f"\n  📐 Debt Capacity Solver — Maximum TLB for {ticker}")
    print(f"     (Given DSCR ≥ {cov['min_dscr']}x and ICR ≥ {cov['min_icr']}x in all years)")
    max_debt = solve_max_tlb_capacity(
        projections[ticker],
        min_dscr = cov['min_dscr'],
        min_icr  = cov['min_icr']
    )
    entry_ebitda = projections[ticker].iloc[0]['EBITDA']
    print(f"     Max TLB Principal:   ${max_debt:>10,.0f}M")
    print(f"     As leverage multiple: {max_debt/entry_ebitda:.2f}x EBITDA")

print("\n✅ Credit statistics computed. Red cells = covenant breach.")


══════════════════════════════════════════════════════════════════════
  CREDIT STATISTICS — Lockheed Martin Corp. (LMT)
  Covenants: DSCR ≥ 1.2x | ICR ≥ 2.5x | Max Leverage 5.5x
══════════════════════════════════════════════════════════════════════

  Option A — TLB Heavy


,EBITDA,Total_Debt,DSCR,ICR,FCCR,Total_Leverage,Senior_Leverage
Year,,,,,,,
Y1,"$9,844M","$44,297M",1.62x,2.37x,2.00x,4.50x,3.50x
Y2,"$10,584M","$43,953M",1.77x,2.58x,2.18x,4.15x,3.22x
Y3,"$11,179M","$43,608M",1.92x,2.75x,2.35x,3.90x,3.02x
Y4,"$11,805M","$43,264M",2.05x,2.93x,2.51x,3.67x,2.83x
Y5,"$12,252M","$42,919M",2.17x,3.07x,2.65x,3.50x,2.70x



  Option B — Balanced


,EBITDA,Total_Debt,DSCR,ICR,FCCR,Total_Leverage,Senior_Leverage
Year,,,,,,,
Y1,"$9,844M","$44,297M",1.57x,2.23x,1.94x,4.50x,2.50x
Y2,"$10,584M","$44,051M",1.71x,2.42x,2.11x,4.16x,2.30x
Y3,"$11,179M","$43,805M",1.85x,2.57x,2.27x,3.92x,2.16x
Y4,"$11,805M","$43,559M",1.97x,2.73x,2.41x,3.69x,2.02x
Y5,"$12,252M","$43,313M",2.09x,2.85x,2.54x,3.54x,1.93x



  Option C — Bond Heavy


...


  📐 Debt Capacity Solver — Maximum TLB for LMT
     (Given DSCR ≥ 1.2x and ICR ≥ 2.5x in all years)
     Max TLB Principal:   $    44,026M
     As leverage multiple: 4.47x EBITDA

══════════════════════════════════════════════════════════════════════
  CREDIT STATISTICS — Palantir Technologies Inc. (PLTR)
  Covenants: DSCR ≥ 1.2x | ICR ≥ 3.0x | Max Leverage 3.0x
══════════════════════════════════════════════════════════════════════

  Option A — Light Leverage


,EBITDA,Total_Debt,DSCR,ICR,FCCR,Total_Leverage,Senior_Leverage
Year,,,,,,,
Y1,"$2,184M","$4,368M",4.56x,5.74x,5.55x,2.00x,1.50x
Y2,"$3,027M","$4,335M",6.42x,8.05x,7.75x,1.43x,1.07x
Y3,"$3,964M","$4,302M",8.53x,10.66x,10.23x,1.08x,0.81x
Y4,"$4,865M","$4,270M",10.63x,13.20x,12.67x,0.88x,0.65x
Y5,"$5,868M","$4,237M",12.94x,16.06x,15.40x,0.72x,0.54x



  Option B — Moderate


,EBITDA,Total_Debt,DSCR,ICR,FCCR,Total_Leverage,Senior_Leverage
Year,,,,,,,
Y1,"$2,184M","$6,006M",3.24x,4.06x,3.94x,2.75x,2.00x
Y2,"$3,027M","$5,962M",4.56x,5.69x,5.51x,1.97x,1.43x
Y3,"$3,964M","$5,919M",6.06x,7.53x,7.27x,1.49x,1.08x
Y4,"$4,865M","$5,875M",7.55x,9.32x,9.00x,1.21x,0.87x
Y5,"$5,868M","$5,831M",9.19x,11.34x,10.93x,0.99x,0.71x



  📐 Debt Capacity Solver — Maximum TLB for PLTR
     (Given DSCR ≥ 1.2x and ICR ≥ 3.0x in all years)
     Max TLB Principal:   $     8,616M
     As leverage multiple: 3.95x EBITDA

✅ Credit statistics computed. Red cells = covenant breach.


In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7: SCENARIO ANALYSIS & STRESS TESTING                     ║
# ║                                                                  ║
# ║  Runs 4 scenarios for each company × each capital structure:    ║
# ║   1. Base Case     — management projections                     ║
# ║   2. Mild Stress   — minor revenue miss + margin compression    ║
# ║   3. Moderate      — recessionary conditions                    ║
# ║   4. Severe        — deep downturn / demand shock               ║
# ║                                                                  ║
# ║  For each combination, we record the MINIMUM DSCR across all   ║
# ║  5 years — because lenders care about the WORST year, not the  ║
# ║  average. A single covenant breach terminates the facility.     ║
# ╚══════════════════════════════════════════════════════════════════╝

# Scenario definitions — overrides applied on top of base case
SCENARIOS = {
    'Base Case': {},   # No overrides — uses company's own assumptions

    'Mild Stress': {
        'revenue_growth': [-0.01, 0.01, 0.02, 0.03, 0.03],
        'ebitda_margin':  None,  # Will be set per company below
    },

    'Moderate Stress': {
        'revenue_growth': [-0.04, -0.02, 0.01, 0.02, 0.02],
        'ebitda_margin':  None,
    },

    'Severe Stress': {
        'revenue_growth': [-0.08, -0.06, -0.02, 0.01, 0.02],
        'ebitda_margin':  None,
    },
}

# Margin compression by scenario (applied as basis point haircut vs. base)
MARGIN_HAIRCUTS = {
    'LMT':  {'Mild Stress': 0.015, 'Moderate Stress': 0.025, 'Severe Stress': 0.040},
    'PLTR': {'Mild Stress': 0.050, 'Moderate Stress': 0.090, 'Severe Stress': 0.140},
    # PLTR gets larger haircuts: software growth cos. are more margin-volatile in downturns
}

# ── BUILD SCENARIO PROJECTIONS ────────────────────────────────────
scenario_projections = {}

for ticker, co_def in COMPANIES.items():
    scenario_projections[ticker] = {}
    base_margins = co_def['projections']['ebitda_margin']

    for scen_name, overrides in SCENARIOS.items():
        scen_overrides = {}

        if scen_name == 'Base Case':
            proj = build_lbo_projection(co_def)
        else:
            haircut = MARGIN_HAIRCUTS[ticker][scen_name]
            stressed_margins = [m - haircut for m in base_margins]
            scen_overrides = {
                'revenue_growth': overrides['revenue_growth'],
                'ebitda_margin':  stressed_margins,
            }
            proj = build_lbo_projection(co_def, scenario_overrides=scen_overrides)

        scenario_projections[ticker][scen_name] = proj

# ── BUILD SCENARIO MATRIX: min DSCR across all structures × scenarios ─
# This powers the heatmap in Cell 8
scenario_matrix = {}

for ticker, co_def in COMPANIES.items():
    scenario_matrix[ticker] = {}
    cov = co_def['covenants']

    for struct_name, tranches in all_structures[ticker].items():
        scenario_matrix[ticker][struct_name] = {}

        for scen_name in SCENARIOS:
            proj = scenario_projections[ticker][scen_name]
            stats = compute_credit_stats(proj, tranches)

            min_dscr  = stats['DSCR'].min()
            min_icr   = stats['ICR'].min()
            max_lev   = stats['Total_Leverage'].max()
            passes    = (min_dscr >= cov['min_dscr'] and
                         min_icr  >= cov['min_icr']  and
                         max_lev  <= cov['max_lev'])

            scenario_matrix[ticker][struct_name][scen_name] = {
                'min_DSCR':    round(min_dscr, 2),
                'min_ICR':     round(min_icr, 2),
                'max_Leverage':round(max_lev, 2),
                'passes':      passes,
            }

# ── PRINT SUMMARY ────────────────────────────────────────────────
print("📊 SCENARIO STRESS TEST RESULTS — Min DSCR (Worst Year)")
print("="*70)

for ticker, co_def in COMPANIES.items():
    print(f"\n  {co_def['name']} ({ticker})  |  Min DSCR threshold: "
          f"{co_def['covenants']['min_dscr']}x")
    print(f"  {'Structure':<35} {'Base':>8} {'Mild':>8} "
          f"{'Moderate':>10} {'Severe':>8}")
    print(f"  {'─'*70}")

    for struct_name in all_structures[ticker]:
        row_vals = []
        for scen in SCENARIOS:
            result = scenario_matrix[ticker][struct_name][scen]
            flag = '✅' if result['passes'] else '❌'
            row_vals.append(f"{flag}{result['min_DSCR']:.2f}x")
        print(f"  {struct_name:<35} "
              f"{row_vals[0]:>9} {row_vals[1]:>9} "
              f"{row_vals[2]:>11} {row_vals[3]:>9}")

print("\n✅ Scenario matrix built. Used by heatmap in next cell.")

📊 SCENARIO STRESS TEST RESULTS — Min DSCR (Worst Year)

  Lockheed Martin Corp. (LMT)  |  Min DSCR threshold: 1.2x
  Structure                               Base     Mild   Moderate   Severe
  ──────────────────────────────────────────────────────────────────────
  Option A — TLB Heavy                   ❌1.62x    ❌1.34x      ❌1.17x    ❌0.91x
  Option B — Balanced                    ❌1.57x    ❌1.29x      ❌1.13x    ❌0.88x
  Option C — Bond Heavy                  ❌1.58x    ❌1.30x      ❌1.14x    ❌0.88x

  Palantir Technologies Inc. (PLTR)  |  Min DSCR threshold: 1.2x
  Structure                               Base     Mild   Moderate   Severe
  ──────────────────────────────────────────────────────────────────────
  Option A — Light Leverage              ✅4.56x    ❌2.94x      ❌2.54x    ❌2.06x
  Option B — Moderate                    ✅3.24x    ❌2.09x      ❌1.80x    ❌1.46x

✅ Scenario matrix built. Used by heatmap in next cell.


In [9]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8: VISUALIZATION SUITE                                     ║
# ║                                                                  ║
# ║  Chart 1: DSCR trajectory — LMT vs PLTR, Option A               ║
# ║  Chart 2: Leverage de-lever path — LMT vs PLTR                  ║
# ║  Chart 3: Heatmap — structure × scenario, pass/fail              ║
# ║  Chart 4: Entry credit quality comparison (bar)                  ║
# ║  Chart 5: Blended cost of debt by structure                      ║
# ╚══════════════════════════════════════════════════════════════════╝

COLORS = {
    'LMT':  {'A': '#003366', 'B': '#0055A5', 'C': '#3388CC'},
    'PLTR': {'A': '#8B0000', 'B': '#CC2200'},
}

# ── CHART 1 & 2: DSCR + LEVERAGE SIDE-BY-SIDE ────────────────────
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Debt Service Coverage Ratio (DSCR)',
        'Total Leverage (Debt / EBITDA)'
    ],
    horizontal_spacing=0.12
)

for ticker, co_def in COMPANIES.items():
    struct_list = list(all_structures[ticker].items())
    opt_letters = ['A', 'B', 'C']

    for j, (struct_name, tranches) in enumerate(struct_list):
        stats = all_credit_stats[ticker][struct_name]
        color = list(COLORS[ticker].values())[j]
        short = f"{ticker} — {struct_name.split('—')[1].strip()}"
        dash  = 'solid' if j == 0 else 'dash'

        fig1.add_trace(go.Scatter(
            x=stats.index, y=stats['DSCR'],
            name=short, mode='lines+markers',
            line=dict(color=color, width=2.5, dash=dash),
            marker=dict(size=7),
            legendgroup=ticker,
        ), row=1, col=1)

        fig1.add_trace(go.Scatter(
            x=stats.index, y=stats['Total_Leverage'],
            name=short, mode='lines+markers',
            line=dict(color=color, width=2.5, dash=dash),
            marker=dict(size=7),
            legendgroup=ticker, showlegend=False,
        ), row=1, col=2)

# Reference lines
fig1.add_hline(y=1.20, line_dash='dot', line_color='#FF4444',
               annotation_text='Min DSCR 1.20x',
               annotation_font=dict(size=10, color='#FF4444'), row=1, col=1)
fig1.add_hline(y=5.50, line_dash='dot', line_color='#FF8800',
               annotation_text='Max Leverage 5.5x (LMT)',
               annotation_font=dict(size=9), row=1, col=2)
fig1.add_hline(y=3.00, line_dash='dot', line_color='#00AA44',
               annotation_text='Target Exit 3.0x',
               annotation_font=dict(size=9), row=1, col=2)

fig1.update_layout(
    title=dict(
        text=(
            '<b>LBO Credit Analysis: Lockheed Martin (LMT) vs. Palantir (PLTR)</b><br>'
            '<sup style="color:gray">Base Case | '
            '⚠️ PLTR analysis is HYPOTHETICAL — '
            'not LBO-feasible at ~137x EV/EBITDA entry valuation</sup>'
        ),
        font=dict(size=15, family='Arial')
    ),
    template='plotly_white',
    height=520, width=1150,
    font=dict(family='Arial', size=11),
    legend=dict(
        x=1.01, y=1.0, bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#CCCCCC', borderwidth=1
    ),
    plot_bgcolor='#FAFAFA',
)
fig1.update_yaxes(title_text='DSCR (x)',       row=1, col=1, gridcolor='#EEEEEE')
fig1.update_yaxes(title_text='Leverage (x)',   row=1, col=2, gridcolor='#EEEEEE')
fig1.show()

# ── CHART 3: HEATMAP — PASS/FAIL BY STRUCTURE × SCENARIO ─────────
for ticker, co_def in COMPANIES.items():
    struct_keys   = list(all_structures[ticker].keys())
    scenario_keys = list(SCENARIOS.keys())

    z_vals   = []
    txt_vals = []

    for sk in struct_keys:
        row_z, row_t = [], []
        for sc in scenario_keys:
            r = scenario_matrix[ticker][sk][sc]
            row_z.append(1.0 if r['passes'] else 0.0)
            status = '✅ PASS' if r['passes'] else '❌ FAIL'
            row_t.append(
                f"DSCR: {r['min_DSCR']:.2f}x<br>"
                f"ICR:  {r['min_ICR']:.2f}x<br>"
                f"{status}"
            )
        z_vals.append(row_z)
        txt_vals.append(row_t)

    clean_structs = [s.split('—')[1].strip() if '—' in s else s
                     for s in struct_keys]

    fig_hm = go.Figure(go.Heatmap(
        z=z_vals,
        x=scenario_keys,
        y=clean_structs,
        text=txt_vals,
        texttemplate='%{text}',
        textfont=dict(size=10, color='black'),
        colorscale=[[0, '#FF6B6B'], [0.5, '#FFD93D'], [1, '#51CF66']],
        showscale=False, zmin=0, zmax=1
    ))

    cov = co_def['covenants']
    lbo_label = '✅ LBO FEASIBLE' if co_def['lbo_viable'] else '⚠️ HYPOTHETICAL ONLY'
    fig_hm.update_layout(
        title=dict(
            text=(
                f'<b>Stress Test Heatmap — {co_def["name"]} ({ticker})</b><br>'
                f'<sup>DSCR ≥ {cov["min_dscr"]}x AND ICR ≥ {cov["min_icr"]}x required '
                f'| {lbo_label}</sup>'
            ),
            font=dict(size=14)
        ),
        template='plotly_white',
        height=300 + len(struct_keys) * 60,
        width=850,
        font=dict(family='Arial', size=11),
        xaxis=dict(tickfont=dict(size=12)),
        yaxis=dict(tickfont=dict(size=11)),
    )
    fig_hm.show()

# ── CHART 4: ENTRY CREDIT QUALITY COMPARISON ─────────────────────
metrics_display = ['EBITDA Margin', 'FCF Margin', 'CapEx % Rev']
lmt_fin = COMPANIES['LMT']['override_financials']
pltr_fin = COMPANIES['PLTR']['override_financials']

lmt_vals  = [
    lmt_fin['base_ebitda']  / lmt_fin['base_revenue'] * 100,
    lmt_fin['base_fcf']     / lmt_fin['base_revenue'] * 100,
    lmt_fin['base_capex']   / lmt_fin['base_revenue'] * 100,
]
pltr_vals = [
    pltr_fin['base_ebitda'] / pltr_fin['base_revenue'] * 100,
    pltr_fin['base_fcf']    / pltr_fin['base_revenue'] * 100,
    pltr_fin['base_capex']  / pltr_fin['base_revenue'] * 100,
]

fig4 = go.Figure()
fig4.add_trace(go.Bar(
    name='Lockheed Martin (LMT)',
    x=metrics_display, y=lmt_vals,
    marker_color='#003366',
    text=[f'{v:.1f}%' for v in lmt_vals],
    textposition='outside', textfont=dict(size=11)
))
fig4.add_trace(go.Bar(
    name='Palantir (PLTR)',
    x=metrics_display, y=pltr_vals,
    marker_color='#CC2200',
    text=[f'{v:.1f}%' for v in pltr_vals],
    textposition='outside', textfont=dict(size=11)
))
fig4.update_layout(
    title=(
        '<b>Credit Quality Profile at LBO Entry: LMT vs. PLTR</b><br>'
        '<sup>Higher EBITDA/FCF margin = better coverage; lower CapEx = less cash drain</sup>'
    ),
    barmode='group', template='plotly_white',
    height=460, width=800,
    font=dict(family='Arial', size=12),
    yaxis=dict(title='% of Revenue', ticksuffix='%', gridcolor='#EEEEEE'),
    legend=dict(x=0.65, y=0.95),
    annotations=[dict(
        text=(
            '<b>Key Insight:</b> PLTR has 3× better margins but its $356B market cap '
            'makes an LBO economically impossible. LMT\'s stable government contracts '
            'make it the only viable LBO candidate of the two.'
        ),
        xref='paper', yref='paper', x=0, y=-0.20,
        font=dict(size=10, color='#444444'),
        showarrow=False, align='left'
    )]
)
fig4.show()

# ── CHART 5: BLENDED COST OF DEBT ────────────────────────────────
fig5 = go.Figure()
all_labels, all_rates, all_bar_colors = [], [], []

for ticker in ['LMT', 'PLTR']:
    for struct_name, tranches in all_structures[ticker].items():
        total_p = sum(t['Principal'] for t in tranches)
        total_i = sum(t['Annual_Cash_Int'] for t in tranches)
        blended = (total_i / total_p * 100) if total_p > 0 else 0
        lbl = f"{ticker}\n{struct_name.split('—')[1].strip()}"
        all_labels.append(lbl)
        all_rates.append(round(blended, 2))
        all_bar_colors.append('#003366' if ticker == 'LMT' else '#CC2200')

fig5.add_trace(go.Bar(
    x=all_labels, y=all_rates,
    marker_color=all_bar_colors,
    text=[f'{r:.2f}%' for r in all_rates],
    textposition='outside',
))
fig5.update_layout(
    title='<b>Blended All-In Cost of Debt by Structure</b>',
    yaxis=dict(title='Blended Rate (%)', ticksuffix='%'),
    template='plotly_white', height=430, width=800,
    font=dict(family='Arial', size=11),
)
fig5.show()

print("✅ All 5 charts rendered.")

✅ All 5 charts rendered.


In [10]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9: EXCEL EXPORT — BANKER-READY WORKBOOK                    ║
# ║                                                                  ║
# ║  Creates LBO_Debt_Capacity_Engine.xlsx with 4 tabs:             ║
# ║   Tab 1: Cover Sheet (summary of the whole analysis)            ║
# ║   Tab 2: LMT Term Sheet (tranche-by-tranche deal terms)         ║
# ║   Tab 3: PLTR Term Sheet                                        ║
# ║   Tab 4: Credit Stats — both companies side-by-side             ║
# ║                                                                  ║
# ║  Color code used throughout:                                     ║
# ║   Dark Navy  (#003366) = Headers                                ║
# ║   Light Blue (#E8F0FE) = Sub-headers                            ║
# ║   Red        (#FFB3B3) = Covenant breach                        ║
# ║   Green      (#B3FFB3) = Strong pass                            ║
# ╚══════════════════════════════════════════════════════════════════╝

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── STYLE HELPERS ─────────────────────────────────────────────────
def hdr(cell, bg='003366', fg='FFFFFF', sz=11, bold=True):
    cell.fill = PatternFill('solid', fgColor=bg)
    cell.font = Font(name='Calibri', bold=bold, color=fg, size=sz)
    cell.alignment = Alignment(horizontal='center', vertical='center',
                                wrap_text=True)
    thin = Side(style='thin')
    cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

def data_cell(cell, bg='FFFFFF', sz=10, bold=False, align='center'):
    cell.fill = PatternFill('solid', fgColor=bg)
    cell.font = Font(name='Calibri', bold=bold, size=sz)
    cell.alignment = Alignment(horizontal=align, vertical='center')
    thin = Side(style='thin')
    cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)


wb = Workbook()

# ═══════════════════════════════════════════════════════════════════
# TAB 1: COVER SHEET
# ═══════════════════════════════════════════════════════════════════
ws_cover = wb.active
ws_cover.title = 'Cover Sheet'
ws_cover.column_dimensions['A'].width = 45
ws_cover.column_dimensions['B'].width = 30
ws_cover.row_dimensions[1].height = 40
ws_cover.row_dimensions[2].height = 25

ws_cover.merge_cells('A1:B1')
ws_cover['A1'] = 'LBO DEBT CAPACITY & CAPITAL STRUCTURE ENGINE'
ws_cover['A1'].font = Font(name='Calibri', bold=True, size=16, color='003366')
ws_cover['A1'].alignment = Alignment(horizontal='center', vertical='center')

ws_cover.merge_cells('A2:B2')
ws_cover['A2'] = 'Leveraged Finance Credit Analysis | Lockheed Martin vs. Palantir'
ws_cover['A2'].font = Font(name='Calibri', italic=True, size=11, color='666666')
ws_cover['A2'].alignment = Alignment(horizontal='center')

cover_rows = [
    ('', ''),
    ('ANALYSIS SCOPE', ''),
    ('Companies Analysed', 'Lockheed Martin (LMT) | Palantir (PLTR)'),
    ('Projection Period', '5-Year LBO Hold (Y1–Y5)'),
    ('Debt Structures Tested', f'{sum(len(c["structures"]) for c in COMPANIES.values())} capital structure options'),
    ('Scenarios', 'Base | Mild Stress | Moderate Stress | Severe Stress'),
    ('', ''),
    ('KEY FINANCIALS (FY2025)', ''),
    ('LMT Revenue',  f'${COMPANIES["LMT"]["override_financials"]["base_revenue"]:,.0f}M'),
    ('LMT EBITDA',   f'${COMPANIES["LMT"]["override_financials"]["base_ebitda"]:,.0f}M'),
    ('LMT FCF',      f'${COMPANIES["LMT"]["override_financials"]["base_fcf"]:,.0f}M'),
    ('PLTR Revenue', f'${COMPANIES["PLTR"]["override_financials"]["base_revenue"]:,.0f}M'),
    ('PLTR EBITDA',  f'${COMPANIES["PLTR"]["override_financials"]["base_ebitda"]:,.0f}M'),
    ('PLTR FCF',     f'${COMPANIES["PLTR"]["override_financials"]["base_fcf"]:,.0f}M'),
    ('', ''),
    ('DEBT PRICING (2025 MARKET)', ''),
    ('Term Loan B',         f'SOFR ({SOFR_RATE:.1%}) + {TLB_SPREAD:.0%} = {SOFR_RATE+TLB_SPREAD:.2%} all-in'),
    ('Senior Secured Notes', f'{SSN_RATE:.2%} fixed'),
    ('High Yield Notes',     f'{HY_RATE:.2%} fixed'),
    ('Mezzanine',            f'{MEZZ_RATE:.2%} fixed'),
    ('', ''),
    ('LBO VIABILITY', ''),
    ('Lockheed Martin',   '✅ FEASIBLE — stable gov contracts, $6.9B FCF'),
    ('Palantir',          '⚠️ NOT FEASIBLE — ~137x EV/EBITDA makes entry economics impossible'),
    ('', ''),
    ('DATA SOURCES', ''),
    ('LMT Financials',   'LMT FY2025 Earnings Release (Jan 28 2026)'),
    ('PLTR Financials',  'PLTR FY2025 (stockanalysis.com | monexa.ai)'),
    ('Debt Pricing',     'Baker McKenzie LevFin 2025 | AFME HY Report Q2 2025'),
]

row = 3
for label, value in cover_rows:
    if label in ('ANALYSIS SCOPE', 'KEY FINANCIALS (FY2025)',
                 'DEBT PRICING (2025 MARKET)', 'LBO VIABILITY', 'DATA SOURCES'):
        ws_cover.merge_cells(f'A{row}:B{row}')
        ws_cover[f'A{row}'] = label
        ws_cover[f'A{row}'].fill = PatternFill('solid', fgColor='003366')
        ws_cover[f'A{row}'].font = Font(name='Calibri', bold=True,
                                        color='FFFFFF', size=11)
        ws_cover[f'A{row}'].alignment = Alignment(horizontal='left',
                                                   indent=1)
    else:
        c1 = ws_cover[f'A{row}']
        c2 = ws_cover[f'B{row}']
        c1.value = label
        c2.value = value
        bg = 'F5F5F5' if row % 2 == 0 else 'FFFFFF'
        data_cell(c1, bg=bg, align='left', bold=bool(label))
        data_cell(c2, bg=bg, align='left')
    row += 1


# ═══════════════════════════════════════════════════════════════════
# FUNCTION: Write a term sheet tab for one company
# ═══════════════════════════════════════════════════════════════════
def write_term_sheet_tab(wb, ticker, company_def):
    ws = wb.create_sheet(f'{ticker} Term Sheet')

    ws.column_dimensions['A'].width = 32
    for c in range(2, 9):
        ws.column_dimensions[get_column_letter(c)].width = 16

    entry_ebitda = projections[ticker].iloc[0]['EBITDA']

    ws.merge_cells('A1:H1')
    ws['A1'] = (f"{company_def['name']} ({ticker}) — "
                f"LBO Term Sheet (Illustrative)")
    ws['A1'].font = Font(name='Calibri', bold=True, size=13, color='003366')
    ws['A1'].alignment = Alignment(horizontal='center')

    ws.merge_cells('A2:H2')
    ws['A2'] = (f"Entry EBITDA: ${entry_ebitda:,.0f}M | "
                f"{'LBO Feasible ✅' if company_def['lbo_viable'] else 'Hypothetical Only ⚠️'}")
    ws['A2'].font = Font(name='Calibri', italic=True, size=10, color='666666')
    ws['A2'].alignment = Alignment(horizontal='center')

    col_hdrs = ['Tranche', 'Size ($M)', 'x EBITDA', 'Rate Type',
                'All-In Rate', 'Amort / Year', 'Tenor', 'Rank']
    for ci, h in enumerate(col_hdrs, 1):
        hdr(ws.cell(row=4, column=ci, value=h))

    row = 5
    for struct_name, tranches in all_structures[ticker].items():
        # Structure sub-header
        ws.merge_cells(f'A{row}:H{row}')
        ws[f'A{row}'] = struct_name
        ws[f'A{row}'].fill = PatternFill('solid', fgColor='E8F0FE')
        ws[f'A{row}'].font = Font(name='Calibri', bold=True,
                                   size=11, color='003366')
        ws[f'A{row}'].alignment = Alignment(horizontal='left', indent=1)
        row += 1

        for t in tranches:
            bg = 'F9F9F9' if row % 2 == 0 else 'FFFFFF'
            vals = [
                t['Name'],
                round(t['Principal'], 0),
                round(t['Principal'] / entry_ebitda, 2),
                'Floating' if t['Is_Floating'] else 'Fixed',
                f"{t['All_In_Rate']:.2%}",
                f"{t['Amort_Pct']:.0%}",
                f"{t['Tenor']}yr",
                t['Rank'],
            ]
            for ci, v in enumerate(vals, 1):
                c = ws.cell(row=row, column=ci, value=v)
                data_cell(c, bg=bg)
            row += 1

        # Totals row
        total_p = sum(t['Principal'] for t in tranches)
        total_i = sum(t['Annual_Cash_Int'] for t in tranches)
        blended = total_i / total_p if total_p > 0 else 0
        totals  = ['TOTAL', round(total_p, 0),
                   round(total_p / entry_ebitda, 2),
                   '', f'{blended:.2%} (blended)', '', '', '']
        for ci, v in enumerate(totals, 1):
            c = ws.cell(row=row, column=ci, value=v)
            data_cell(c, bg='DDEEFF', bold=True)
        row += 2

    return ws


# ═══════════════════════════════════════════════════════════════════
# FUNCTION: Write credit stats tab for one company
# ═══════════════════════════════════════════════════════════════════
def write_credit_stats_tab(wb, ticker, company_def):
    ws = wb.create_sheet(f'{ticker} Credit Stats')
    ws.column_dimensions['A'].width = 30
    for c in range(2, 8):
        ws.column_dimensions[get_column_letter(c)].width = 14

    cov = company_def['covenants']

    ws.merge_cells('A1:G1')
    ws['A1'] = (f"{company_def['name']} ({ticker}) — "
                f"Credit Statistics | Base Case")
    ws['A1'].font = Font(name='Calibri', bold=True, size=13, color='003366')
    ws['A1'].alignment = Alignment(horizontal='center')

    ws.merge_cells('A2:G2')
    ws['A2'] = (f"Covenants: DSCR ≥ {cov['min_dscr']}x | "
                f"ICR ≥ {cov['min_icr']}x | Max Leverage {cov['max_lev']}x")
    ws['A2'].font = Font(name='Calibri', italic=True, size=10, color='444444')
    ws['A2'].alignment = Alignment(horizontal='center')

    row = 4
    metric_defs = [
        ('Revenue ($M)',         'Revenue',         '${:,.0f}'),
        ('EBITDA ($M)',          'EBITDA',          '${:,.0f}'),
        ('UFCF ($M)',            'UFCF',            '${:,.0f}'),
        ('Total Debt ($M)',      'Total_Debt',      '${:,.0f}'),
        ('Cash Interest ($M)',   'Cash_Interest',   '${:,.0f}'),
        ('Amortization ($M)',    'Amortization',    '${:,.0f}'),
        ('Total Debt Service',   'Debt_Service',    '${:,.0f}'),
        ('DSCR',                 'DSCR',            '{:.2f}x'),
        ('Interest Coverage',    'ICR',             '{:.2f}x'),
        ('FCCR',                 'FCCR',            '{:.2f}x'),
        ('Total Leverage',       'Total_Leverage',  '{:.2f}x'),
        ('Senior Leverage',      'Senior_Leverage', '{:.2f}x'),
    ]

    for struct_name, stats in all_credit_stats[ticker].items():
        # Structure header
        ws.merge_cells(f'A{row}:G{row}')
        ws[f'A{row}'] = struct_name
        ws[f'A{row}'].fill = PatternFill('solid', fgColor='003366')
        ws[f'A{row}'].font = Font(name='Calibri', bold=True,
                                   color='FFFFFF', size=11)
        ws[f'A{row}'].alignment = Alignment(horizontal='left', indent=1)
        row += 1

        # Year column headers
        yr_hdrs = ['Metric'] + list(stats.index)
        for ci, h in enumerate(yr_hdrs, 1):
            hdr(ws.cell(row=row, column=ci, value=str(h)), bg='4472C4')
        row += 1

        # Metric rows
        proj_df = projections[ticker]
        full_stats = stats.copy()
        full_stats['Revenue'] = proj_df['Revenue']

        for label, field, fmt in metric_defs:
            bg = 'F5F5F5' if row % 2 == 0 else 'FFFFFF'
            c0 = ws.cell(row=row, column=1, value=label)
            data_cell(c0, bg=bg, bold=True, align='left')

            for ci, yr in enumerate(stats.index, 2):
                if field == 'Revenue':
                    val = proj_df.loc[yr, 'Revenue'] if yr in proj_df.index else 0
                else:
                    val = stats.loc[yr, field] if field in stats.columns else 0

                c = ws.cell(row=row, column=ci, value=round(float(val), 2))
                data_cell(c, bg=bg)

                # Color-code covenant breaches
                if field == 'DSCR' and float(val) < cov['min_dscr']:
                    c.fill = PatternFill('solid', fgColor='FFB3B3')
                elif field == 'ICR' and float(val) < cov['min_icr']:
                    c.fill = PatternFill('solid', fgColor='FFB3B3')
                elif field == 'Total_Leverage' and float(val) > cov['max_lev']:
                    c.fill = PatternFill('solid', fgColor='FFDDB3')
            row += 1
        row += 2

    return ws


# ── BUILD ALL TABS ─────────────────────────────────────────────────
for ticker, co_def in COMPANIES.items():
    write_term_sheet_tab(wb, ticker, co_def)
    write_credit_stats_tab(wb, ticker, co_def)

# ── SAVE AND DOWNLOAD ──────────────────────────────────────────────
OUTPUT_PATH = '/content/LBO_Debt_Capacity_Engine_LMT_vs_PLTR.xlsx'
wb.save(OUTPUT_PATH)
print(f"✅ Excel workbook saved: {OUTPUT_PATH}")
print("   Downloading to your computer...")
files.download(OUTPUT_PATH)
print("✅ Download complete.")
print("\n   Tabs created:")
for sheet in wb.sheetnames:
    print(f"    → {sheet}")


✅ Excel workbook saved: /content/LBO_Debt_Capacity_Engine_LMT_vs_PLTR.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download complete.

   Tabs created:
    → Cover Sheet
    → LMT Term Sheet
    → LMT Credit Stats
    → PLTR Term Sheet
    → PLTR Credit Stats
